# **The Goal: Investigate Muon-pair production in the ATLAS-LHC data**
In this notebook we present several aspects of muon-pair production over a large mass range from around **1 GeV** up to **several TeV**.

* We will first discuss how to process the data, the details of the data format and the stored parameters.
* We will show some examples how to access the content and investigate the properties of the parameters.
* We will proceed with a very basic selection of oppositely charged muon pairs  (a muon and an anti-muon: $\mu^+ \mu^-$)
* and an algorithm to calculate the invariant mass of our $\mu^+ \mu^-$ pair.




The principal process we investigate are $p p$ collisions at ATLAS/LHC which result in a 
$\mu^+ \mu^-$ pair in the final state:

$pp \rightarrow  \mu^+ \mu^-  (+ X...)$


See [example event display](https://twiki.cern.ch/twiki/pub/AtlasPublic/EventDisplayRun2Physics/JiveXML_327265_117236869-YX-RZ-RZ-EventInfo-2017-07-10-09-38-22.png)

In such $pp$ collisions there are always many other particles (+X...) which are produced and
and recorded.

Depending on the $\mu^+ \mu^-$ pair mass different production mechanisms contribute:
* In the range $1 .. 10$ GeV the $\mu^+ \mu^-$ pairs mostly originate from decays of mesons, such as $J /\Psi$ or $\Upsilon$. These mesons are typically created in the hadronisation following strong interaction processes between partons in the primary protons.
* At higher masses the production is dominated by the so-called Drell-Yan process: a quark and an anti-quark annihilate into a photon or Z boson which then decays into a $\mu^+ \mu^-$ pair
  $$ q \bar{q} \rightarrow Z/\gamma \rightarrow  \mu^+ \mu^- $$
* Moreover, also Higgs bosons can decay into $\mu^+ \mu^-$ pairs, i.e. the process
  $$pp \rightarrow H \rightarrow \mu^+ \mu^-$$
  contributes as well, although with a rather small cross-section
* Potentially further high-mass bosons (e.g. $Z'$) could exist and show up as a resonance at high masses, it is important to investigate the high-mass spectrum for such contributions.


The data we are using is part of the [ATLAS open data project](https://opendata.atlas.cern/).


**Contents:**  
* setup
* data content
* 4-momentum and invariant mass
* data processing and event selection
* mass spectra plots



In [1]:
import ROOT
# enable interactive plotting features
%jsroot on 

In [2]:
import os

path = "/project/cip/2025-SS-Duckeck/data/opendata/2muons/Data/"
os.listdir(path)

['data16_periodA.2muons.root',
 'data15_periodE.2muons.root',
 'data15_periodH.2muons.root',
 'data15_periodG.2muons.root',
 'data16_periodL.2muons.root',
 'data16_periodC.2muons.root',
 'data16_periodE.2muons.root',
 'data16_periodG.2muons.root',
 'data16_periodD.2muons.root',
 'data16_periodK.2muons.root',
 'data16_periodI.2muons.root',
 'data16_periodF.2muons.root',
 'data15_periodD.2muons.root',
 'data16_periodB.2muons.root',
 'data15_periodF.2muons.root']

In [3]:
# ATLAS Open Data directory
# path = "https://cernbox.cern.ch/remote.php/dav/public-files/kjdB71TXAlY06ee/2muons/Data/'
# path = "/project/etp/gduckeck/data/opendata/2muons/Data/"
path = "/project/cip/2025-SS-Duckeck/data/opendata/2muons/Data/"
samples_list = [
    "data15_periodD.2muons.root",
    "data15_periodF.2muons.root",
    "data15_periodE.2muons.root",
    "data15_periodG.2muons.root",
    "data15_periodH.2muons.root",
    "data16_periodA.2muons.root",
    "data16_periodB.2muons.root",
    "data16_periodC.2muons.root",
    "data16_periodD.2muons.root",
    "data16_periodE.2muons.root",
    "data16_periodF.2muons.root",
    "data16_periodG.2muons.root",
    "data16_periodI.2muons.root",
    "data16_periodK.2muons.root",
    "data16_periodL.2muons.root",
]

In [4]:
# Create a chain containing the analysis trees
chain = ROOT.TChain("analysis")
# Add all our input files to the chain
nfiles = 2 # restrict #-files
for file_name in samples_list[:nfiles]:
    chain.Add(path+file_name)


In [5]:
chain.GetEntries()

833676

In [ ]:
# check content
chain.Print()

#### Track coordinates
ATLAS/LHC has a special (sphere-like) coordinate system, namely  `pt, phi, eta`, expressed in cartesian coordinates these are defined as

$$ p_x = p_T \cos \phi $$
$$ p_y = p_T \sin \phi $$
$$ p_z = p_T / \tan( 2 \arctan( \exp(-\eta) )) $$ 


`pt, phi, eta` are choosen because they provide better match for detector parameters and kinematics.

In [7]:
# histogram of pt
c = ROOT.TCanvas("c","myCanvas",500,400)
chain.Draw("lep_pt")
c.Draw()

In [8]:
# provide helper function

def nbdraw( myobj, myopts="", mysize=(500,400) ):
    "helper function for drawing in notebook"
    c = ROOT.TCanvas("myc","myCanvas",*mysize);
    myobj.Draw(myopts)
    c.Draw()  
    return c


In [9]:
# use pre-defined histo
h1 = ROOT.TH1F("pt1","Muon pT", 100, 0, 200)
nbdraw(chain,"lep_pt >> pt1")

### Exercise - 1
* Plot histogram of pt of 2nd and 3rd lepton
* Plot histogram of pt of all leptons
* make scatter plot  of pt of 1st vs 2nd lepton
* investigate other parameters (eta, phi, other particle types, ...)

### Invariant mass of muon pair
For the next step we need to calculate the invariant mass of our muon pair

This is rather straightforward when using Lorentz-Vectors, i.e. we need to create
Lorentz-Vectors for our 2 muons:
$$p_L = (E, \vec{p})$$
We could do this manually by calculating the  transformation from our `pt, phi, eta` coordinates to  cartesian coordinates `p_x, p_y, p_z`:
$$ p_x = p_T \cos \phi $$
$$ p_y = p_T \sin \phi $$
$$ p_z = p_T / \tan( 2 \arctan( \exp(-\eta) )) $$ 

However, more convenient is to use the `TLorentzVector` class which provides this functionality:

In [17]:
lv = ROOT.TLorentzVector()
pt, eta, phi, m = (9.158, 0.67559, -3.134, 0.)
lv.SetPtEtaPhiM(pt, eta, phi, 0)
lv.Print()

### Process the data chain

In [24]:
# To save memory and go a little bit faster, we will make two Lorentz vectors
# here and re-use them during the loop over events
Muon_1 = ROOT.TLorentzVector()
Muon_2 = ROOT.TLorentzVector()

# To make this run a bit faster, we will disable any branches we don't need
chain.SetBranchStatus('*',0)

lepkeys = ["lep_pt", "lep_eta", "lep_phi", "lep_e", "lep_type", "lep_n", "lep_charge"]
# Now let's enable exactly the branches we want
for variable in lepkeys:
    chain.SetBranchStatus(variable,1)


stop_early = 200002

# create histogram for invariant mass
hist = ROOT.TH1F("h_M_Hmm","Dimuon invariant-mass ; Invariant Mass m_{mumu} [GeV] ; Events",300,5,160)


for n,ev in enumerate(chain):
    # Check if we should stop
    if n>stop_early and stop_early>0:
        break
    # Print the number of events we have processed so far
    if n%10000==0:
        print(f'Processing event {n}')

        # Cut #1: At least 2 leptons
    if ev.lep_n >= 2:
        
        # Cut #2: Leptons with opposite charge
        if (ev.lep_charge[0] != ev.lep_charge[1]):
            
            # Cut #3: both are muons (id = 13)
            if (ev.lep_type[0] == 13 and ev.lep_type[1] == 13):
        
                Muon_1.SetPtEtaPhiM(ev.lep_pt[0],
                          ev.lep_eta[0],
                          ev.lep_phi[0],
                                  0)
                Muon_2.SetPtEtaPhiM(ev.lep_pt[1],
                          ev.lep_eta[1],
                          ev.lep_phi[1],
                                  1)
                # Add the two TLorentz vectors
                Muon_12 = Muon_1 + Muon_2
                # fill inv mass of 2 Lvecs in histo
                hist.Fill(Muon_12.M())

Processing event 0
Processing event 10000
Processing event 20000
Processing event 30000
Processing event 40000
Processing event 50000
Processing event 60000
Processing event 70000
Processing event 80000
Processing event 90000
Processing event 100000
Processing event 110000
Processing event 120000
Processing event 130000
Processing event 140000
Processing event 150000
Processing event 160000
Processing event 170000
Processing event 180000
Processing event 190000
Processing event 200000


Warning in <TROOT::Append>: Replacing existing TH1: h_M_Hmm (Potential memory leak).


In [25]:
nbdraw(hist)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: myc


In [16]:
ev.lep_pt[0], ev.lep_eta[0], ev.lep_phi[0], 0

(9.158788681030273, 0.6755910515785217, -3.1347525119781494, 0)

### Exercise 3 -- investigate mass spectrum

make several histograms to investigate
* low mass region (<20)
* medium range(70-110)
* around the Higgs (110-140)
* at high values (200-2000)

Discuss observations